# Global mCH quantification (supporting)

Part of the **[Fig. 3 chapter](fig3.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{outdir}clustering/merged/L1pre/5kCG_lsi.h5ad'`  ·  _raw embedding_
- `f'{outdir}clustering/merged/L1/5kCG100k3C_embed.h5ad'`  ·  _embedding h5ad_
- `f'{indir}merged/ALL_lambda.csv.gz'`  ·  _lambda control_


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [2]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [3]:
indir = '/data/ENTEx/'
outdir = '/mnt/disks/jupyter/home/jzhou_salk_edu/sky_workdir/ENTEx/'


In [4]:
tmp = anndata.read_h5ad(f'{outdir}clustering/merged/L1pre/5kCG_lsi.h5ad')
tmp


In [5]:
adata = anndata.read_h5ad(f'{outdir}clustering/merged/L1/5kCG100k3C_embed.h5ad')
adata


In [6]:
lam = pd.read_csv(f'{indir}merged/ALL_lambda.csv.gz', header=0, index_col=0)
lam

In [7]:
leg = np.sort(meta['Tissue'].unique())
leg

In [8]:
for xx, ax in zip(leg, axes.flatten()):
    tmp = meta.loc[meta['Tissue'] == xx]
    cov = np.log10(tmp['cov'] + 1)
    lim = [min([np.percentile(tmp['ratio'], 1), np.percentile(tmp['mCHFrac'], 1)]), max([np.percentile(tmp['ratio'], 99), np.percentile(tmp['mCHFrac'], 99)])]
    diff = 0.05 * (lim[1] - lim[0])

In [9]:
selcol = ['Tissue', 'Sample', 'Donor', 
          'Cis/Trans', 'CisLongContact', 'FinalmCReads', 
          'mCHmC', 'mCHCov', 'mCHFrac', 
          'mCGmC', 'mCGCov', 'mCGFrac', 
          'mCCCmC', 'mCCCCov', 'mCCCFrac']
meta = pd.concat([tmp.obs.loc[adata.obs.index, selcol], lam.loc[adata.obs.index]], axis=1)
meta['L1'] = adata.obs['L1'].copy() 
meta

In [10]:
mch_df = meta.groupby('L1')[['mCHmC', 'mCHCov', 'mc', 'cov']].sum()
mch_df['LambdamCHFrac'] = mch_df['mc'] / mch_df['cov']
mch_df['BulkmCHFrac'] = mch_df['mCHmC'] / mch_df['mCHCov']
mch_df['AvemCHFrac'] = meta.groupby('L1')['mCHFrac'].mean()
mch_df.sort_values('AvemCHFrac') 

In [11]:
leg = np.sort(mch_df.index)
colordict = {xx: yy for xx, yy in zip(leg, sns.color_palette('tab20', len(leg)))}
ylim = [[0.056, 0.061], [0.005, 0.021]]
d = 0.5
kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12, linestyle='none', color='k', mec='k', mew=1, clip_on=False)